# UCDEmbed — Low-Resource UCD-vector vs Byte/Subword Study (A100)

**The new direction.** The byte study (`byteembed_lowresource_a100.ipynb`) feeds raw **UTF-8 bytes**
through a ByT5 encoder. This notebook explores a representation we have **not** tried before: the
student reads **Unicode Character Database (UCD) property vectors** per codepoint instead of bytes —
`General_Category`, `Canonical_Combining_Class`, `Bidi_Class`, script/block, binary flags, plus a
CANINE-style hashed-codepoint *identity* channel. The transformer body is the **same ByT5 encoder**;
only the input front end changes (fed via `inputs_embeds`), so the iso-compute / parameter-allocation
comparison stays clean.

**Why it might matter (the hypotheses):**
1. **One position per character** (not per byte) → removes the UTF-8 multibyte tax for Indic/Ethiopic
   scripts (Tamil byte-seq 9.9× → ~1×), so sequence length sits *between* subword and byte.
2. **Cross-lingual parameter sharing** → one "combining acute" / "Letter, other" feature reused across
   every language instead of disjoint subword rows; the input table is tiny.
3. **Structured generalisation** → a codepoint unseen in training still arrives with a valid
   script/category/combining structure (transferable signal a raw byte cannot carry).

Same teacher (**SONAR**, 1024-d, 200 langs), same 9 languages (te/ta/mr/am/ha/rw + en/zh/ar anchors),
same uniform battery (SIB-200 · Belebele · FLORES-1012 · STS · MIRACL), and the **same cached teacher
targets** as the byte study — so UCD trains against identical supervision. Everything is resumable:
results save after each model, checkpoints cache model+optimizer, the teacher pass is cached. Run
top-to-bottom; do the smoke cell first.

### 1. GPU check — confirm you're on an A100 (Runtime → Change runtime type → A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps
Imports work from the repo root even if the editable install is skipped.

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. SONAR teacher (no fairseq2 needed)
SONAR loads by default via the **fairseq2-free HF port** `cointegrated/SONAR_200_text_encoder`
(1024-d, validated to match official SONAR). No install, no torch change. **This is the reliable
default.** The optional official `sonar-space` path is identical to the byte notebook — uncomment, run,
**Restart Runtime**, re-run from cell 2 — but the HF port is recommended.

**The cached targets are shared with the byte study**: same teacher name + same tag, so if you already
ran `byteembed_lowresource_a100.ipynb` with the same data on this Drive, this notebook reuses that exact
teacher pass (no re-embedding) and UCD trains on identical supervision.

In [ ]:
from transformers import AutoTokenizer
from transformers.models.m2m_100.modeling_m2m_100 import M2M100Encoder
_ = AutoTokenizer.from_pretrained('cointegrated/SONAR_200_text_encoder')
print('SONAR (HF port) reachable — teacher will be SONAR (1024-d), no fairseq2 needed')

# --- OPTIONAL: the literal official sonar-space (fairseq2). Run, then RESTART RUNTIME, re-run cell 2.
# !pip install -q torch==2.9.1 --index-url https://download.pytorch.org/whl/cu126
# !pip install -q fairseq2 --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.9.1/cu126
# !pip install -q sonar-space

### 4. Persist results + checkpoints to Drive
So a Colab disconnect doesn't lose cached teacher targets / checkpoints / results. **Skip this cell**
to run on ephemeral disk. Point `PERSIST` at the SAME folder you used for the byte study to reuse its
cached teacher targets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'   # SAME folder as the byte study -> shared cache
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)
# from huggingface_hub import login; login()   # uncomment + run if you hit HF rate limits

### 5. Smoke test (~5 min) — validate the WHOLE pipeline first
3 langs (am/rw/en), one tiny UCD student, tiny eval. If this prints a table, everything works
(teacher load → balanced data → cached targets → UCD featurizer → train → SIB/Belebele/FLORES/STS/MIRACL
→ save). Confirm the teacher log says 'official SONAR' or 'SONAR via HF port' (both 1024-d), not LaBSE.

In [ ]:
from byte_embed.run_ucd import run
_ = run(smoke=True, pooling='mean', out='results/ucd_lowresource_smoke.json')   # matches the preliminary config

### 6. Tokenization-efficiency table (fast, no training)
The motivation metric: subword token tax vs byte UTF-8 tax (vs English, on parallel FLORES-1012). The
UCD representation is **1 position per character**, so it removes the byte UTF-8 multibyte penalty shown
in the `byte_seq_x` column (the Indic 7–10× blow-up) while staying tokenizer-free.

In [ ]:
from byte_embed.efficiency import fertility_table, print_fertility
from byte_embed.config import STUDY_LANGS
print_fertility(fertility_table(STUDY_LANGS))

### 7. Full study — UCD students (small/base/large) — PRELIMINARY config
**Mean pooling** + the **original 10k / 13k / 15k step schedule**, matched to the byteembed *preliminary*
run (`byte_lowresource.json`) so the UCD numbers drop straight into the same table. Same 9 languages,
SONAR teacher, and battery (SIB · Belebele · FLORES · STS · MIRACL); reuses the byte study's **identical
cached teacher targets** (no re-embedding), so UCD vs byte vs subword is directly comparable.

Runs **sequentially** (so the loss/eval print **live** in this cell — no buffered logs), ~a few hours,
resumable. `compare=False` — the byte/subword preliminary numbers already live in `byte_lowresource.json`;
set `compare=True` only if you want to re-train them alongside UCD here.

In [ ]:
# PRELIMINARY config — mean pooling + original 10k/13k/15k schedule, matched to byte_lowresource.json.
# Sequential run() => loss + eval print LIVE in this cell (no buffered logs). Reuses the byte study's
# cached SONAR targets, so UCD trains on identical supervision and drops straight into the same table.
from byte_embed.run_ucd import run
_ = run(
    out='results/ucd_lowresource.json',
    pooling='mean',                                          # match the byteembed PRELIMINARY run
    steps={'small': 10000, 'base': 13000, 'large': 15000},   # original preliminary schedule
    compare=False,                                           # byte/subword already in byte_lowresource.json
)

### 8. Results — table
Per-model SIB / Belebele / FLORES / STS / MIRACL with the input-vs-transformer parameter split (the
UCD input table is tiny). If you ran `compare=True`, the UCD−byte and UCD−subword deltas at matched size
print too.

In [ ]:
import json
from byte_embed.run_ucd import _summary
res = json.load(open('results/ucd_lowresource.json'))
_summary(res)

### 9. Download results
Already on Drive if you ran cell 4. Otherwise grab the results JSON here.

In [ ]:
from google.colab import files
files.download('results/ucd_lowresource.json')